# Data missingness check (actual pipeline input)

This notebook loads price data using the same code-path as the pipeline (`run/config.py:load_price_data`) and summarizes missingness / coverage per ticker.

- Set `MARKET` to `"cn"` or `"us"`.
- By default it checks the full range loaded by the pipeline: `cfg.data_split.train_start` → `cfg.data_split.test_end`.


In [10]:
import os
import sys
import polars as pl
from pathlib import Path
from IPython.display import display

def _find_project_root() -> Path:
    """Find the `01_15_new_qlib` project root so `import run...` works in notebooks."""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "run").is_dir() and (p / "run_pipeline.py").is_file():
            return p
        cand = p / "01_15_new_qlib"
        if (cand / "run").is_dir() and (cand / "run_pipeline.py").is_file():
            return cand
    raise RuntimeError(
        "Could not locate project root. Run this notebook from within the repo, or adjust sys.path manually."
    )


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

# --- Choose market to match your experiment ---
MARKET = "cn"  # "cn" or "us"
os.environ["MARKET"] = MARKET

# Optional: override load range (None = pipeline defaults)
OVERRIDE_START = None  # e.g., "2015-01-01"
OVERRIDE_END = None    # e.g., "2025-12-31"

def _norm_date(s: str) -> str:
    return str(s).replace("-", "")

def _missing_expr(col: str) -> pl.Expr:
    # Null + NaN (NaN only applies for float columns)
    return (pl.col(col).is_null() | pl.col(col).is_nan()).cast(pl.Int64).sum()

def _trading_rows_expr() -> pl.Expr:
    # volume>0 as a proxy for a "trading" row
    return (pl.col("volume").fill_null(0).cast(pl.Float64) > 0).cast(pl.Int64).sum()

def _missing_expr_on_trading_days(col: str) -> pl.Expr:
    is_trading = pl.col("volume").fill_null(0).cast(pl.Float64) > 0
    is_missing = (pl.col(col).is_null() | pl.col(col).is_nan())
    return (is_trading & is_missing).cast(pl.Int64).sum()


PROJECT_ROOT: /home/dgu/fin/01_15_new_qlib


In [11]:
from run.config import load_rd_config, load_price_data

cfg = load_rd_config()
df = load_price_data(cfg, start_time=OVERRIDE_START, end_time=OVERRIDE_END)

print("MARKET:", MARKET)
print("shape:", df.shape)
print("date range:", df.select(pl.col("timestamp").min()).item(), "~", df.select(pl.col("timestamp").max()).item())
print("n_tickers:", df.select(pl.col("ticker").n_unique()).item())
print("n_dates:", df.select(pl.col("timestamp").n_unique()).item())

# Duplicate check (ticker, date)
n_dups = (
    df.group_by(["ticker", "timestamp"])\
      .len()\
      .filter(pl.col("len") > 1)
).height
print("duplicate (ticker,timestamp) rows:", n_dups)


[1792754:MainThread](2026-02-08 06:43:54,222) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[1792754:MainThread](2026-02-08 06:43:54,229) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[1792754:MainThread](2026-02-08 06:43:54,231) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/home/dgu/.qlib/qlib_data/cn_data')}


📊 Loading data from Qlib...
✅ Qlib initialized: /home/dgu/.qlib/qlib_data/cn_data (region=cn, market=csi500)
📊 Loading csi500 data from 2015-01-01 to 2025-12-31...
✅ Loaded 1,335,655 rows from 1292 instruments
📊 Final DataFrame shape: (1335655, 11)
   Date range: 20150105 ~ 20251229
MARKET: cn
shape: (1335655, 11)
date range: 20150105 ~ 20251229
n_tickers: 1292
n_dates: 2672
duplicate (ticker,timestamp) rows: 0


In [12]:
def summarize_missingness(panel: pl.DataFrame, label: str) -> pl.DataFrame:
    n_all_dates = panel.select(pl.col("timestamp").n_unique()).item()
    out = (
        panel.group_by("ticker")
        .agg(
            pl.len().alias("n_rows"),
            pl.col("timestamp").n_unique().alias("n_dates"),
            _trading_rows_expr().alias("n_trading_rows"),
            _missing_expr("open").alias("open_missing"),
            _missing_expr("high").alias("high_missing"),
            _missing_expr("low").alias("low_missing"),
            _missing_expr("close").alias("close_missing"),
            _missing_expr("volume").alias("volume_missing"),
            _missing_expr_on_trading_days("close").alias("close_missing_trading"),
        )
        .with_columns(
            (pl.col("n_dates") / pl.lit(n_all_dates)).alias("date_coverage"),
            (pl.lit(n_all_dates) - pl.col("n_dates")).alias("missing_dates"),
            (
                pl.col("open_missing")
                + pl.col("high_missing")
                + pl.col("low_missing")
                + pl.col("close_missing")
                + pl.col("volume_missing")
            ).alias("ohlcv_missing_total"),
        )
        .sort(["close_missing", "close_missing_trading", "missing_dates"], descending=[True, True, True])
    )

    n_tickers = out.height
    n_close_complete = out.filter(pl.col("close_missing") == 0).height
    n_close_complete_trading = out.filter(pl.col("close_missing_trading") == 0).height
    n_ohlcv_complete = out.filter(pl.col("ohlcv_missing_total") == 0).height
    print(f"\n=== {label} ===")
    print("n_tickers:", n_tickers)
    print("close complete tickers:", n_close_complete)
    print("close complete (trading rows only) tickers:", n_close_complete_trading)
    print("OHLCV complete tickers:", n_ohlcv_complete)
    print("top missing tickers:")
    display(out.head(20))
    return out


# Full window (what Stage1 formula_df is computed on)
full_stats = summarize_missingness(df, "FULL (pipeline input)")

# In-sample / out-of-sample windows used downstream
is_start, is_end = _norm_date(cfg.data_split.train_start), _norm_date(cfg.data_split.train_end)
oos_start, oos_end = _norm_date(cfg.data_split.test_start), _norm_date(cfg.data_split.test_end)

df_is = df.filter((pl.col("timestamp") >= is_start) & (pl.col("timestamp") <= is_end))
df_oos = df.filter((pl.col("timestamp") >= oos_start) & (pl.col("timestamp") <= oos_end))

is_stats = summarize_missingness(df_is, f"IS (train): {cfg.data_split.train_start} ~ {cfg.data_split.train_end}")
oos_stats = summarize_missingness(df_oos, f"OOS (test): {cfg.data_split.test_start} ~ {cfg.data_split.test_end}")



=== FULL (pipeline input) ===
n_tickers: 1292
close complete tickers: 668
close complete (trading rows only) tickers: 1292
OHLCV complete tickers: 668
top missing tickers:


ticker,n_rows,n_dates,n_trading_rows,open_missing,high_missing,low_missing,close_missing,volume_missing,close_missing_trading,date_coverage,missing_dates,ohlcv_missing_total
str,u32,u32,i64,i64,i64,i64,i64,i64,i64,f64,u32,i64
"""SH689009""",851,851,162,689,689,689,689,689,0,0.318488,1821,3445
"""SH600759""",1335,1335,966,369,369,369,369,369,0,0.499626,1337,1845
"""SZ300134""",1579,1579,1245,334,334,334,334,334,0,0.590943,1093,1670
"""SZ300088""",2430,2430,2099,331,331,331,331,331,0,0.909431,242,1655
"""SH600645""",1704,1704,1379,325,325,325,325,325,0,0.637725,968,1625
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SZ000748""",499,499,263,236,236,236,236,236,0,0.186751,2173,1180
"""SZ002219""",363,363,128,235,235,235,235,235,0,0.135853,2309,1175
"""SH600654""",363,363,140,223,223,223,223,223,0,0.135853,2309,1115



=== IS (train): 2015-01-01 ~ 2019-12-31 ===
n_tickers: 909
close complete tickers: 359
close complete (trading rows only) tickers: 909
OHLCV complete tickers: 359
top missing tickers:


ticker,n_rows,n_dates,n_trading_rows,open_missing,high_missing,low_missing,close_missing,volume_missing,close_missing_trading,date_coverage,missing_dates,ohlcv_missing_total
str,u32,u32,i64,i64,i64,i64,i64,i64,i64,f64,u32,i64
"""SH600759""",1219,1219,850,369,369,369,369,369,0,1.0,0,1845
"""SZ300134""",1219,1219,885,334,334,334,334,334,0,1.0,0,1670
"""SZ300088""",1219,1219,888,331,331,331,331,331,0,1.0,0,1655
"""SH600645""",1219,1219,894,325,325,325,325,325,0,1.0,0,1625
"""SZ000979""",974,974,671,303,303,303,303,303,0,0.799016,245,1515
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SZ002219""",363,363,128,235,235,235,235,235,0,0.297785,856,1175
"""SH600654""",363,363,140,223,223,223,223,223,0,0.297785,856,1115
"""SH600655""",974,974,752,222,222,222,222,222,0,0.799016,245,1110



=== OOS (test): 2021-01-01 ~ 2025-12-31 ===
n_tickers: 890
close complete tickers: 804
close complete (trading rows only) tickers: 890
OHLCV complete tickers: 804
top missing tickers:


ticker,n_rows,n_dates,n_trading_rows,open_missing,high_missing,low_missing,close_missing,volume_missing,close_missing_trading,date_coverage,missing_dates,ohlcv_missing_total
str,u32,u32,i64,i64,i64,i64,i64,i64,i64,f64,u32,i64
"""SH689009""",851,851,162,689,689,689,689,689,0,0.703306,359,3445
"""SZ002670""",1210,1210,1180,30,30,30,30,30,0,1.0,0,150
"""SZ000401""",1085,1085,1063,22,22,22,22,22,0,0.896694,125,110
"""SZ000528""",609,609,588,21,21,21,21,21,0,0.503306,601,105
"""SH601198""",1093,1093,1073,20,20,20,20,20,0,0.903306,117,100
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SZ000826""",242,242,232,10,10,10,10,10,0,0.2,968,50
"""SZ002013""",359,359,349,10,10,10,10,10,0,0.296694,851,50
"""SZ000656""",360,360,350,10,10,10,10,10,0,0.297521,850,50


In [13]:
# If you want to run with ONLY close-complete tickers:
close_complete = full_stats.filter(pl.col("close_missing") == 0).select("ticker")
tickers_close_complete = close_complete.to_series().to_list()

print("close-complete ticker count:", len(tickers_close_complete))
print("sample:", tickers_close_complete[:20])

# Example filtered frame
df_close_complete = df.filter(pl.col("ticker").is_in(tickers_close_complete))
print("filtered shape:", df_close_complete.shape)


close-complete ticker count: 668
sample: ['SZ300014', 'SH688047', 'SZ300413', 'SH600467', 'SH600439', 'SZ002083', 'SZ002052', 'SH600197', 'SZ000886', 'SH601011', 'SZ300347', 'SH600239', 'SH600616', 'SH600740', 'SH600830', 'SZ000981', 'SZ002581', 'SH600981', 'SH600346', 'SZ002493']
filtered shape: (558290, 11)


In [14]:
complete = full_stats.filter(
    (pl.col("date_coverage") == 1.0) & (pl.col("ohlcv_missing_total") == 0)
).select("ticker")
complete

ticker
str
"""SH600563"""
"""SZ002273"""
